# GPU Optimization Guide для Multi-Agent Q-Learning

Полный гайд по оптимизации вычислений на GPU для системы многоагентного обучения с подкреплением. Включает практические примеры, бенчмарки и техники оптимизации.

## Оглавление
1. Проверка GPU и базовая конфигурация
2. Минимальный бенчмарк CPU vs GPU
3. Оптимизация передачи данных
4. Векторизация операций
5. Смешанная точность (Mixed Precision)
6. Профилирование и анализ производительности
7. Интеграция GPU в существующий код

## Раздел 1: Проверка доступности GPU и базовая конфигурация

Проверяем наличие GPU и настраиваем окружение для оптимальной работы.

In [ ]:
import torch
import numpy as np
import time
import matplotlib.pyplot as plt
import psutil
import os

# Проверяем доступность GPU
print("="*60)
print("ПРОВЕРКА ДОСТУПНОСТИ GPU")
print("="*60)

cuda_available = torch.cuda.is_available()
print(f"CUDA доступна: {cuda_available}")

if cuda_available:
    device = torch.device("cuda")
    print(f"Текущее устройство: {torch.cuda.get_device_name(0)}")
    print(f"Версия CUDA: {torch.version.cuda}")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
    
    # Характеристики памяти
    props = torch.cuda.get_device_properties(0)
    print(f"\nХарактеристики GPU:")
    print(f"  - Глобальная память: {props.total_memory / 1e9:.2f} GB")
    print(f"  - Количество CUDA ядер: {props.multi_processor_count * 128}")
    print(f"  - Тактовая частота: {props.clock_rate / 1e9:.2f} GHz")
    
    # Текущее использование памяти
    print(f"\nИспользование памяти GPU:")
    print(f"  - Выделено: {torch.cuda.memory_allocated() / 1e6:.2f} MB")
    print(f"  - Зарезервировано: {torch.cuda.memory_reserved() / 1e6:.2f} MB")
else:
    device = torch.device("cpu")
    print("GPU недоступна, используем CPU")
    
print(f"\nУстройство по умолчанию: {device}")
print(f"PyTorch версия: {torch.__version__}")

## Раздел 2: Минимальный бенчмарк CPU vs GPU

Сравниваем производительность основных операций на CPU и GPU.

In [ ]:
def benchmark_matrix_operations(sizes, n_iterations=10):
    """Бенчмарк матричных операций на CPU и GPU"""
    cpu_times = []
    gpu_times = []
    
    for size in sizes:
        # Создаём матрицы
        A_cpu = np.random.randn(size, size)
        B_cpu = np.random.randn(size, size)
        
        # CPU benchmark: матричное умножение
        A_torch_cpu = torch.from_numpy(A_cpu).float()
        B_torch_cpu = torch.from_numpy(B_cpu).float()
        
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        start = time.time()
        for _ in range(n_iterations):
            C_cpu = torch.mm(A_torch_cpu, B_torch_cpu)
        cpu_time = (time.time() - start) / n_iterations
        cpu_times.append(cpu_time)
        
        # GPU benchmark (если доступна)
        if torch.cuda.is_available():
            A_gpu = A_torch_cpu.cuda()
            B_gpu = B_torch_cpu.cuda()
            
            torch.cuda.synchronize()
            start = time.time()
            for _ in range(n_iterations):
                C_gpu = torch.mm(A_gpu, B_gpu)
            torch.cuda.synchronize()
            gpu_time = (time.time() - start) / n_iterations
            gpu_times.append(gpu_time)
        else:
            gpu_times.append(cpu_time)  # Если GPU нет, используем CPU время
    
    return cpu_times, gpu_times

# Запускаем бенчмарк
print("\n" + "="*60)
print("БЕНЧМАРК: Матричное умножение")
print("="*60)

matrix_sizes = [100, 500, 1000, 2000]
cpu_times, gpu_times = benchmark_matrix_operations(matrix_sizes, n_iterations=20)

print("\nРезультаты (время в секундах):")
print(f"{'Размер':<12} {'CPU':<15} {'GPU':<15} {'Ускорение':<12}")
print("-" * 54)

for size, cpu_t, gpu_t in zip(matrix_sizes, cpu_times, gpu_times):
    speedup = cpu_t / gpu_t if gpu_t > 0 else 1.0
    print(f"{size:<12} {cpu_t:.6f}s{'':<8} {gpu_t:.6f}s{'':<8} {speedup:.2f}x")

# Визуализация
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.loglog(matrix_sizes, cpu_times, 'o-', label='CPU', linewidth=2, markersize=8)
plt.loglog(matrix_sizes, gpu_times, 's-', label='GPU', linewidth=2, markersize=8)
plt.xlabel('Размер матрицы (NxN)')
plt.ylabel('Время (секунды)')
plt.title('Сравнение скорости матричного умножения')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
speedups = [cpu / gpu if gpu > 0 else 1.0 for cpu, gpu in zip(cpu_times, gpu_times)]
plt.plot(matrix_sizes, speedups, 'g^-', linewidth=2, markersize=8)
plt.xlabel('Размер матрицы (NxN)')
plt.ylabel('Ускорение (раз)')
plt.title('GPU ускорение относительно CPU')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Раздел 3: Оптимизация передачи данных (PCI-E пропускная способность)

Демонстрируем важность эффективной передачи данных между CPU и GPU.

In [ ]:
def benchmark_data_transfer(sizes, n_iterations=100):
    """Бенчмарк передачи данных CPU <-> GPU"""
    transfer_times_cpu_to_gpu = []
    transfer_times_gpu_to_cpu = []
    
    print("\n" + "="*60)
    print("БЕНЧМАРК: Передача данных CPU <-> GPU")
    print("="*60)
    
    for size in sizes:
        data_cpu = np.random.randn(size, size).astype(np.float32)
        
        if not torch.cuda.is_available():
            print("GPU недоступна, пропускаем тест...")
            break
        
        # CPU to GPU transfer
        torch.cuda.synchronize()
        start = time.time()
        for _ in range(n_iterations):
            data_gpu = torch.from_numpy(data_cpu).cuda()
        torch.cuda.synchronize()
        transfer_cpu_to_gpu = (time.time() - start) / n_iterations
        transfer_times_cpu_to_gpu.append(transfer_cpu_to_gpu)
        
        # GPU to CPU transfer
        data_gpu = torch.from_numpy(data_cpu).cuda()
        torch.cuda.synchronize()
        start = time.time()
        for _ in range(n_iterations):
            data_cpu_back = data_gpu.cpu().numpy()
        torch.cuda.synchronize()
        transfer_gpu_to_cpu = (time.time() - start) / n_iterations
        transfer_times_gpu_to_cpu.append(transfer_gpu_to_cpu)
        
        # Calculate bandwidth
        data_size_gb = (size * size * 4) / 1e9  # 4 bytes per float32
        bandwidth_cpu_to_gpu = data_size_gb / transfer_cpu_to_gpu
        bandwidth_gpu_to_cpu = data_size_gb / transfer_gpu_to_cpu
        
        print(f"\nМатрица {size}x{size} ({data_size_gb:.4f} GB):")
        print(f"  CPU → GPU: {transfer_cpu_to_gpu*1000:.3f} ms ({bandwidth_cpu_to_gpu:.2f} GB/s)")
        print(f"  GPU → CPU: {transfer_gpu_to_cpu*1000:.3f} ms ({bandwidth_gpu_to_cpu:.2f} GB/s)")
    
    return transfer_times_cpu_to_gpu, transfer_times_gpu_to_cpu

# Запускаем тест
if torch.cuda.is_available():
    data_sizes = [100, 500, 1000, 2000]
    transfer_cpu_gpu, transfer_gpu_cpu = benchmark_data_transfer(data_sizes)
    
    # Визуализация
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(data_sizes, transfer_cpu_gpu, 'o-', label='CPU → GPU', linewidth=2, markersize=8)
    plt.plot(data_sizes, transfer_gpu_cpu, 's-', label='GPU → CPU', linewidth=2, markersize=8)
    plt.xlabel('Размер матрицы (NxN)')
    plt.ylabel('Время передачи (сек)')
    plt.title('Время передачи данных')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    bandwidth_cpu_gpu = [(s*s*4/1e9)/t for s, t in zip(data_sizes, transfer_cpu_gpu)]
    bandwidth_gpu_cpu = [(s*s*4/1e9)/t for s, t in zip(data_sizes, transfer_gpu_cpu)]
    plt.plot(data_sizes, bandwidth_cpu_gpu, 'o-', label='CPU → GPU', linewidth=2, markersize=8)
    plt.plot(data_sizes, bandwidth_gpu_cpu, 's-', label='GPU → CPU', linewidth=2, markersize=8)
    plt.xlabel('Размер матрицы (NxN)')
    plt.ylabel('Пропускная способность (GB/s)')
    plt.title('PCI-E пропускная способность')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("GPU недоступна")

## Раздел 4: Векторизация и фьюзинг операций на GPU

Показываем выгоду от замены циклов на векторизованные операции.

In [ ]:
print("\n" + "="*60)
print("ПРИМЕР 1: Награды (Reward Calculation)")
print("="*60)

# Пример: расчёт наград для агентов (из нашего проекта)
# Неоптимизированная версия: цикл по агентам

def compute_rewards_loop(strategies, adj_matrix, degrees, b, c):
    """Неоптимизированная версия с циклом"""
    rewards = []
    for i in range(len(strategies)):
        neighbors = np.nonzero(adj_matrix[i])[0]
        neighbor_contribution = np.sum(strategies[neighbors])
        reward = b * neighbor_contribution - c * strategies[i] * degrees[i]
        rewards.append(reward)
    return np.array(rewards)

# Оптимизированная версия: матричные операции
def compute_rewards_vectorized(strategies, adj_matrix, degrees, b, c):
    """Оптимизированная версия с векторизацией"""
    neighbor_sum = adj_matrix @ strategies
    rewards = b * neighbor_sum - c * strategies * degrees
    return rewards

# Бенчмарк
n_agents = 1000
strategies = np.random.randint(0, 2, n_agents).astype(np.float32)
degrees = np.random.randint(1, 20, n_agents).astype(np.float32)
adj_matrix = np.random.randint(0, 2, size=(n_agents, n_agents)).astype(np.float32)
b, c = 3.0, 1.0

# CPU benchmark
start = time.time()
for _ in range(100):
    rewards_loop = compute_rewards_loop(strategies, adj_matrix, degrees, b, c)
cpu_loop_time = time.time() - start

start = time.time()
for _ in range(100):
    rewards_vec = compute_rewards_vectorized(strategies, adj_matrix, degrees, b, c)
cpu_vec_time = time.time() - start

print(f"\nCPU (NumPy):")
print(f"  С циклом (loop):       {cpu_loop_time:.4f}s")
print(f"  С векторизацией:       {cpu_vec_time:.4f}s")
print(f"  Ускорение:             {cpu_loop_time/cpu_vec_time:.2f}x")

# GPU benchmark (если доступна)
if torch.cuda.is_available():
    strategies_gpu = torch.from_numpy(strategies).cuda()
    degrees_gpu = torch.from_numpy(degrees).cuda()
    adj_gpu = torch.from_numpy(adj_matrix).cuda()
    
    def compute_rewards_gpu(strategies, adj_matrix, degrees, b, c):
        neighbor_sum = adj_matrix @ strategies
        rewards = b * neighbor_sum - c * strategies * degrees
        return rewards
    
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(100):
        rewards_gpu = compute_rewards_gpu(strategies_gpu, adj_gpu, degrees_gpu, b, c)
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    
    print(f"\nGPU (PyTorch CUDA):")
    print(f"  С векторизацией:       {gpu_time:.4f}s")
    print(f"  GPU ускорение vs CPU:  {cpu_vec_time/gpu_time:.2f}x")
    
    # Визуализация
    methods = ['Loop\n(CPU)', 'Vectorized\n(CPU)', 'Vectorized\n(GPU)']
    times = [cpu_loop_time, cpu_vec_time, gpu_time]
    colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(methods, times, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    
    # Добавляем значения на столбцы
    for bar, t in zip(bars, times):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{t:.4f}s',
                ha='center', va='bottom', fontweight='bold')
    
    plt.ylabel('Время выполнения (сек)', fontsize=12)
    plt.title(f'Сравнение методов расчёта наград\n({n_agents} агентов, 100 итераций)', fontsize=14, fontweight='bold')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("\nGPU недоступна")

## Раздел 5: Смешанная точность (Mixed Precision) и управление памятью

Демонстрируем использование смешанной точности (FP16 + FP32) для ускорения и экономии памяти.

In [ ]:
print("\n" + "="*60)
print("СМЕШАННАЯ ТОЧНОСТЬ (Mixed Precision)")
print("="*60)

if torch.cuda.is_available():
    # Размеры матриц для теста
    matrix_size = 2000
    
    # Создаём данные
    A_fp32 = torch.randn(matrix_size, matrix_size, dtype=torch.float32, device='cuda')
    B_fp32 = torch.randn(matrix_size, matrix_size, dtype=torch.float32, device='cuda')
    
    A_fp16 = A_fp32.half()
    B_fp16 = B_fp32.half()
    
    # FP32 benchmark
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(100):
        C_fp32 = torch.mm(A_fp32, B_fp32)
    torch.cuda.synchronize()
    time_fp32 = (time.time() - start) / 100
    
    # FP16 benchmark
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(100):
        C_fp16 = torch.mm(A_fp16, B_fp16)
    torch.cuda.synchronize()
    time_fp16 = (time.time() - start) / 100
    
    # Сравнение памяти
    mem_fp32 = (A_fp32.element_size() * A_fp32.nelement() + B_fp32.element_size() * B_fp32.nelement()) / 1e6
    mem_fp16 = (A_fp16.element_size() * A_fp16.nelement() + B_fp16.element_size() * B_fp16.nelement()) / 1e6
    
    print(f"\nМатричное умножение {matrix_size}x{matrix_size}:")
    print(f"  FP32 (32-bit float):")
    print(f"    - Время: {time_fp32*1000:.3f} ms")
    print(f"    - Память: {mem_fp32:.2f} MB")
    print(f"  FP16 (16-bit float):")
    print(f"    - Время: {time_fp16*1000:.3f} ms")
    print(f"    - Память: {mem_fp16:.2f} MB")
    print(f"\n  Ускорение (FP32 → FP16): {time_fp32/time_fp16:.2f}x")
    print(f"  Экономия памяти: {mem_fp32/mem_fp16:.2f}x")
    
    # Визуализация
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # График скорости
    precisions = ['FP32\n(32-bit)', 'FP16\n(16-bit)']
    times = [time_fp32*1000, time_fp16*1000]
    colors = ['#FF6B6B', '#4ECDC4']
    
    bars1 = axes[0].bar(precisions, times, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    axes[0].set_ylabel('Время (ms)', fontsize=12)
    axes[0].set_title('Скорость матричного умножения', fontsize=13, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
    
    for bar, t in zip(bars1, times):
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{t:.1f}ms',
                    ha='center', va='bottom', fontweight='bold')
    
    # График памяти
    memory = [mem_fp32, mem_fp16]
    bars2 = axes[1].bar(precisions, memory, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    axes[1].set_ylabel('Память (MB)', fontsize=12)
    axes[1].set_title('Использование памяти', fontsize=13, fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)
    
    for bar, m in zip(bars2, memory):
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{m:.0f}MB',
                    ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Очистка памяти
    del A_fp32, B_fp32, A_fp16, B_fp16, C_fp32, C_fp16
    torch.cuda.empty_cache()
else:
    print("GPU не доступна")

## Раздел 6: Профилирование и поиск узких мест

Используем встроенные инструменты PyTorch для профилирования GPU операций.

In [ ]:
if torch.cuda.is_available():
    print("\n" + "="*60)
    print("ПРОФИЛИРОВАНИЕ GPU ОПЕРАЦИЙ")
    print("="*60)
    
    def profile_game_operations():
        """Профилирование операций игры"""
        n_agents = 500
        
        # Создаём данные
        strategies = torch.randint(0, 2, (n_agents,), dtype=torch.float32, device='cuda')
        adj_matrix = torch.randint(0, 2, (n_agents, n_agents), dtype=torch.float32, device='cuda')
        degrees = torch.sum(adj_matrix, dim=1)
        
        # Операции, которые часто выполняются в игре
        def game_step():
            # 1. Расчёт состояний
            states = adj_matrix @ strategies
            
            # 2. Расчёт наград
            rewards = 3.0 * (adj_matrix @ strategies) - 1.0 * strategies * degrees
            
            # 3. Обновление стратегий
            strategies_new = torch.where(states > 0.5, torch.ones_like(strategies), torch.zeros_like(strategies))
            
            return states, rewards, strategies_new
        
        # Профилирование с torch.profiler
        print("\nПрофилирование основной игровой операции...")
        
        with torch.profiler.profile(
            activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
            record_shapes=True,
            profile_memory=True
        ) as prof:
            for _ in range(10):
                game_step()
        
        # Вывод результатов
        print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))
        
        # Анализ памяти
        print("\n" + "-"*60)
        print("АНАЛИЗ ИСПОЛЬЗОВАНИЯ ПАМЯТИ")
        print("-"*60)
        
        allocated = torch.cuda.memory_allocated() / 1e6
        reserved = torch.cuda.memory_reserved() / 1e6
        
        print(f"Выделено памяти:  {allocated:.2f} MB")
        print(f"Зарезервировано:  {reserved:.2f} MB")
        print(f"Свободно в резерве: {(reserved - allocated):.2f} MB")
        
        # Статистика памяти
        print(f"\nЖизненный цикл памяти:")
        stats = torch.cuda.memory_stats()
        print(f"  Всего выделено:     {stats['allocated_bytes.all.peak'] / 1e6:.2f} MB")
        print(f"  Максимальное использование: {stats['reserved_bytes.all.peak'] / 1e6:.2f} MB")
    
    profile_game_operations()
    
    # Очистка
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
else:
    print("GPU не доступна для профилирования")

## Раздел 7: Интеграция GPU в существующий код

Практические советы по интеграции GPU оптимизаций в проект.

In [ ]:
print("\n" + "="*60)
print("БЫСТРЫЙ СТАРТ: Использование GPU модулей")
print("="*60)

print("""
СПОСОБ 1: Замена импортов (minimal changes)
────────────────────────────────────

Вместо:
    from learner import QLearner
    from reward_model import PPReward
    from game_launcher import MonteKarloPairGame

Используйте:
    from gpu_learner import GPUQLearner
    from gpu_reward_model import GPUPPReward
    from gpu_game_launcher import GPUMonteKarloPairGame

ПРИМЕР КОДА:
────────────────────────────────────
""")

# Показываем пример использования
code_example = '''
# Импорты GPU модулей
from gpu_learner import GPUQLearner
from gpu_reward_model import GPUPPReward
from gpu_game_launcher import GPUMonteKarloPairGame
from graph_structure import StarGraph

# Параметры
N_NODES = 100
B = 3.0
C = 1.0

# Создаём граф
graph = StarGraph(N_NODES)

# Создаём GPU-оптимизированных агентов
learners = [
    GPUQLearner(
        action_space_size=2,
        learning_rate=0.2,
        discount_factor=0.9,
        strategy='boltzmann',
        max_states=N_NODES+1
    )
    for _ in range(N_NODES)
]

# Создаём GPU-оптимизированную модель наград
reward_model = GPUPPReward(b=B, c=C)

# Запускаем GPU-оптимизированную игру
game = GPUMonteKarloPairGame(graph, learners, reward_model, k_anchors=1)

# Синхронизация и очистка (важно!)
import torch
torch.cuda.synchronize()
torch.cuda.empty_cache()

# Тренировка
for episode in range(1000):
    game.round()
    cooperation_rate = float(torch.mean(game.strategies).item())
    print(f"Episode {episode}: Cooperation rate = {cooperation_rate:.3f}")
'''

print(code_example)

print("\n" + "="*60)
print("СОВЕТЫ ПО ОПТИМИЗАЦИИ")
print("="*60)

tips = """
1. УПРАВЛЕНИЕ ПАМЯТЬЮ:
   ✓ Используйте torch.cuda.empty_cache() для освобождения памяти
   ✓ Мониторьте использование памяти с torch.cuda.memory_allocated()
   ✓ Используйте контекстные менеджеры для автоматической очистки

2. СИНХРОНИЗАЦИЯ:
   ✓ Всегда вызывайте torch.cuda.synchronize() перед измерением времени
   ✓ Без синхронизации GPU операции выполняются асинхронно

3. ТИПЫ ДАННЫХ:
   ✓ Используйте float32 для стабильности обучения
   ✓ Для матричных операций можно использовать float16 (быстрее)
   ✓ Избегайте частого преобразования типов

4. ПЕРЕДАЧА ДАННЫХ:
   ✓ Минимизируйте операции CPU ↔ GPU
   ✓ Передавайте данные один раз, выполняйте множество операций
   ✓ Используйте батчи для эффективной загрузки

5. РАЗМЕР БАТЧА:
   ✓ Больший размер батча = лучше использование GPU
   ✓ Найдите оптимальный размер для вашего устройства
   ✓ Мониторьте использование памяти при увеличении размера

6. ПРОФИЛИРОВАНИЕ:
   ✓ Используйте torch.profiler для поиска узких мест
   ✓ Ищите операции, которые выполняются долго
   ✓ Приоритизируйте оптимизацию самых медленных операций
"""

print(tips)

# Сводка улучшений
print("\n" + "="*60)
print("ОЖИДАЕМЫЕ УЛУЧШЕНИЯ")
print("="*60)

improvements = {
    "Операция": ["Матричное умножение", "Наград расчёты", "Полная игра (1 раунд)"],
    "CPU": ["~5-10ms", "~2-3ms", "~10-15ms"],
    "GPU": ["~0.5-1ms", "~0.1-0.2ms", "~1-2ms"],
    "Ускорение": ["5-10x", "15-20x", "5-10x"],
}

import pandas as pd
df = pd.DataFrame(improvements)
print(df.to_string(index=False))

print("\n" + "="*60)
print("ГОТОВО К ИСПОЛЬЗОВАНИЮ!")
print("="*60)